# 🧬 Digital Twin Evaluation
Train and evaluate the LSTM thermal spike predictor on rack telemetry data.
Target: ≥90% accuracy in predicting spikes (>78°C) 10 minutes ahead.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✓')

In [ ]:
# Load telemetry data
df = pd.read_csv('../data/rack_telemetry_sample.csv')
print(f'Loaded {len(df):,} telemetry rows')
print(f'Spike rate: {df["thermal_spike"].mean()*100:.1f}%')
df.describe()

In [ ]:
# Build sliding window sequences for LSTM
SEQ_LEN    = 30    # 15-min history
FEATURES   = ['temp_celsius','power_watts','coolant_flow_l_min','pue','hour_of_day']
TARGET     = 'thermal_spike'

# Normalise features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])

# Add sin/cos time encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)

INPUT_FEATURES = ['temp_celsius','power_watts','coolant_flow_l_min','pue','hour_sin','hour_cos']

def build_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[INPUT_FEATURES].iloc[i:i+seq_len].values)
        y.append(data[TARGET].iloc[i+seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = build_sequences(df, SEQ_LEN)
print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'Positive (spike) rate: {y.mean()*100:.1f}%')

In [ ]:
# Train/validation/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

In [ ]:
# Train with ThermalTwinModel (requires PyTorch)
try:
    from src.digital_twin.simulator import ThermalTwinModel
    model = ThermalTwinModel(hidden_size=128, num_layers=2, seq_len=SEQ_LEN)

    EPOCHS     = 20
    BATCH_SIZE = 128
    train_losses, val_accs = [], []

    for epoch in range(1, EPOCHS + 1):
        # Mini-batch training
        idx = np.random.permutation(len(X_train))
        epoch_losses = []
        for start in range(0, len(X_train), BATCH_SIZE):
            batch_idx = idx[start:start+BATCH_SIZE]
            loss = model.train_step(X_train[batch_idx], y_train[batch_idx])
            epoch_losses.append(loss)

        # Validation
        val_metrics = model.evaluate(X_val, y_val)
        avg_loss    = np.mean(epoch_losses)
        train_losses.append(avg_loss)
        val_accs.append(val_metrics['accuracy'])

        if epoch % 5 == 0:
            print(f'Epoch {epoch:02d}/{EPOCHS} | Loss={avg_loss:.4f} | '
                  f'Val Acc={val_metrics["accuracy"]*100:.1f}% | '
                  f'F1={val_metrics["f1"]*100:.1f}%')

    model.save('../checkpoints/twin.pt')
    print('\nModel saved ✓')

except ImportError:
    print('PyTorch not available — install torch to run LSTM training.')
    # Simulate results for demonstration
    train_losses = [0.42 - i*0.012 for i in range(20)]
    val_accs     = [0.72 + i*0.009 for i in range(20)]
    print('Using simulated training curves for visualisation.')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Digital Twin LSTM Training', fontsize=13, fontweight='bold')

axes[0].plot(train_losses, color='#ff6b35', linewidth=2)
axes[0].set_title('Training Loss (BCE)'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot([a*100 for a in val_accs], color='#00e5ff', linewidth=2)
axes[1].axhline(90, color='#39ff14', linestyle='--', label='90% target')
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f'Final validation accuracy: {val_accs[-1]*100:.1f}%')

In [ ]:
# Physics-based simulator demo
from src.digital_twin.simulator import ThermalPhysicsSimulator

sim = ThermalPhysicsSimulator()

# Simulate 20 steps (10 minutes) starting at elevated temperature
horizon = sim.predict_horizon(temp_c=72.0, power_w=820.0,
                               coolant_flow=85.0, horizon_steps=20)

temps  = [h['next_temp'] for h in horizon]
spikes = [h['thermal_spike'] for h in horizon]

plt.figure(figsize=(10, 4))
plt.plot(temps, color='#ff6b35', linewidth=2, marker='o', markersize=4, label='Predicted temp')
plt.axhline(78, color='#ffb800', linestyle='--', label='Spike threshold 78°C')
plt.axhline(85, color='#ff2d2d', linestyle='--', label='Critical 85°C')
plt.fill_between(range(len(temps)), 78, [max(78,t) for t in temps],
                 alpha=0.2, color='#ff2d2d', label='Spike zone')
plt.title('Digital Twin: 10-Minute Thermal Forecast')
plt.xlabel('Step (×30s)'); plt.ylabel('Temperature (°C)')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Predicted spikes in 10 min: {sum(spikes)} / {len(spikes)} steps')